In [1]:
!pip install --upgrade pyreadstat --quiet

In [2]:
!pip install --upgrade pandas --quiet

In [3]:
import pandas as pd
import numpy as np
import pyreadstat

In [4]:
import os
import requests
import zipfile
import shutil
import subprocess

In [5]:
url = "https://www.mds.gov.br/webarquivos/publicacao/sagi/microdados/02_bolsa_familia/aibf2.zip"
zip_path = "aibf2.zip"
extract_dir = "raw"
os.makedirs(extract_dir, exist_ok=True)

In [6]:
# ==============================
# 1. baixar dados
# ==============================

print("Baixando arquivo...")
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

Baixando arquivo...


In [7]:
# ==============================
# 2. extrair zip e apagá-lo
# ==============================

print("Extraindo ZIP inicial...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

os.remove(zip_path)
print("ZIP inicial removido.")

Extraindo ZIP inicial...
ZIP inicial removido.


In [8]:
# ==============================
# 3. instalar o RAR (manual + verificação)
# ==============================

def check_rar():
    try:
        subprocess.run(["unrar"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True
    except FileNotFoundError:
        return False

if not check_rar():
    print("\n⚠️ RAR não encontrado.")
    print("Instale manualmente:")
    print("- Windows: https://www.win-rar.com/")
    print("- Linux (Debian/Ubuntu): sudo apt install unrar")
    print("- macOS (brew): brew install unrar\n")

In [9]:
# ==============================
# 4. extrair 'Microdados para analise de impacto.rar'
# ==============================

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".rar") and "Microdados" in file:
            rar_path = os.path.join(root, file)
            print(f"Extraindo {file}...")

            subprocess.run(["unrar", "x", rar_path, root], check=False)

            os.remove(rar_path)
            print(f"{file} removido.")

Extraindo Microdados para analise descritiva_dados intermediarios.rar...

UNRAR 7.22 freeware      Copyright (c) 1993-2026 Alexander Roshal

Extracting from raw/Microdados para analise descritiva_dados intermediarios.rar

Extracting  raw/Section 16.zip                                        OK 
Extracting  raw/_Consumption Aggregates.zip                           OK 
Extracting  raw/_Misc.zip                                             OK 
Extracting  raw/01_Sintaxes.zip                                       OK 
Extracting  raw/02_Tabelas.zip                                        OK 
Extracting  raw/Section 03.zip                                        OK 
Extracting  raw/Section 04.zip                                        OK 
Extracting  raw/Section 05.zip                                        OK 
Extracting  raw/Section 06.zip                                        OK 
Extracting  raw/Section 07.zip                                        OK 
Extracting  raw/Section 08.zip        


ERROR: Bad archive raw/__MACOSX/._Microdados para analise descritiva_dados intermediarios.rar


In [10]:
# ==============================
# 5.  localizar variaveis.zip e education.zip e extrair esses zips
# ==============================

zip_targets = ["Section 05.zip", "Painel_Longitudinal_2005-2009.zip"]

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith('.zip'):
            zip_file_path = os.path.join(root, file)
            if file in zip_targets:
                print(f"Extraindo {file}...")
    
                with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                    zip_ref.extractall(root)

            os.remove(zip_file_path)
            print(f"{file} removido.")

print("\n✅ Processo concluído!")

Section 08.zip removido.
Section 09.zip removido.
Extraindo Painel_Longitudinal_2005-2009.zip...
Painel_Longitudinal_2005-2009.zip removido.
prenatal care.zip removido.
Dados_Brutos_2009.zip removido.
education.zip removido.
anthropometry.zip removido.
descricao das variaveis.zip removido.
social capital.zip removido.
_Consumption Aggregates.zip removido.
womens decision making.zip removido.
_Misc.zip removido.
01_Sintaxes.zip removido.
vaccinations.zip removido.
02_Tabelas.zip removido.
Dados_Brutos_2005.zip removido.
Section 15.zip removido.
birthweight.zip removido.
Section 14.zip removido.
labor supply.zip removido.
Section 16.zip removido.
Section 03.zip removido.
Section 07.zip removido.
Section 13.zip removido.
Section 12.zip removido.
Section 06.zip removido.
child labor.zip removido.
Section 10.zip removido.
Section 04.zip removido.
Extraindo Section 05.zip...
Section 05.zip removido.
Section 11.zip removido.

✅ Processo concluído!


In [11]:
# ==============================
# 6. manter apenas arquivos específicos
# ==============================

arquivos_permitidos_day_04 = [
    "hhlink_2005_2009.dta",
    os.path.join("Section 5", "data2005.sav"),
    os.path.join("Section 5", "data2009.sav")
]

print("Filtrando arquivos...")

arquivos_mantidos_day04 = []

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        caminho_completo = os.path.join(root, file)
        caminho_relativo = os.path.relpath(caminho_completo, extract_dir)

        if caminho_relativo in arquivos_permitidos_day_04:
            arquivos_mantidos_day04.append(caminho_completo)
        else:
            os.remove(caminho_completo)

Filtrando arquivos...


In [12]:
# ==============================
# 7. criar pasta "day 01" e mover arquivos
# ==============================

destino = os.path.join(extract_dir, "day 04")
os.makedirs(destino, exist_ok=True)

print("Movendo arquivos selecionados...")

for arquivo in arquivos_mantidos_day04:
    nome_arquivo = os.path.basename(arquivo)
    novo_caminho = os.path.join(destino, nome_arquivo)
    shutil.move(arquivo, novo_caminho)

Movendo arquivos selecionados...


In [13]:
# ==============================
# 8. remover diretórios vazios
# ==============================

print("Removendo diretórios vazios...")

for root, dirs, files in os.walk(extract_dir, topdown=False):
    for d in dirs:
        dir_path = os.path.join(root, d)
        if not os.listdir(dir_path):
            os.rmdir(dir_path)

print("✅ Organização concluída! Arquivos estão em 'day 04'.")

Removendo diretórios vazios...
✅ Organização concluída! Arquivos estão em 'day 04'.


In [14]:
# ======================================================================
# 1. hhlink — pré-tratamento 2005 e classificação T/C (de Brauw et al.)
# ======================================================================
link, _ = pyreadstat.read_dta("raw/day 04/hhlink_2005_2009.dta")

COLS_LINK = ["cod_dtm", "w2005_chave", "w2005_treat_control",
             "w2005_T1", "w2005_C1", "w2005_C2",
             "w2005_a3_reg", "w2005_a1_uf",
             "w2005_num_age0_age15", "w2005_inc_pc_hh_wunemp",
             "w2005_educ_m", "w2005_educ_f",
             "w2005_age_m", "w2005_age_f", "w2005_sitdom"]
link = link[COLS_LINK].copy()

# grupo (3 categorias)
def classifica(row):
    if row["w2005_T1"] == 1: return "T1"
    if row["w2005_C1"] == 1: return "C1"
    if row["w2005_C2"] == 1: return "C2"
    return np.nan
link["grupo"] = link.apply(classifica, axis=1)
link = link.dropna(subset=["grupo"])
print("Domicílios por grupo:", link["grupo"].value_counts().to_dict())

Domicílios por grupo: {'C2': 7992, 'T1': 4068, 'C1': 3351}


In [15]:
# ======================================================================
# 2. Desfechos educacionais — agregar por domicílio (SEM p_attend7)
# ======================================================================
def agrega_educ(df, ano, chave_col):
    # crianças 6-17 anos
    idade_col = "s02a09_ida" if "s02a09_ida" in df.columns else "age"
    df = df[df[idade_col].between(6, 17)].copy()

    # dropout e grade_now podem não existir em 2005 — checar
    agg_dict = {"n_criancas": (chave_col, "size"),
                "attend_med": ("attend_now", "mean")}
    if "dropout" in df.columns:
        agg_dict["dropout_med"] = ("dropout", "mean")
    if "grade_now" in df.columns:
        agg_dict["grade_now_med"] = ("grade_now", "mean")

    agg = df.groupby(chave_col).agg(**agg_dict).reset_index()
    agg["ano"] = ano
    agg = agg.rename(columns={chave_col: "chave_local"})
    return agg

In [16]:
d05, _ = pyreadstat.read_sav("raw/day 04/data2005.sav")
d09, _ = pyreadstat.read_sav("raw/day 04/data2009.sav")

edu05 = agrega_educ(d05, 2005, "CHAVE")        # 2005 usa w2005_chave
edu05 = edu05.rename(columns={"chave_local": "w2005_chave"})
edu09 = agrega_educ(d09, 2009, "cod_dtm")      # 2009 já usa cod_dtm
edu09 = edu09.rename(columns={"chave_local": "cod_dtm"})

# mapear edu05 para cod_dtm via link
edu05 = edu05.merge(link[["w2005_chave", "cod_dtm"]],
                    on="w2005_chave", how="inner")
edu05 = edu05.drop(columns=["w2005_chave"])

# empilhar 2005 e 2009
desfechos = pd.concat([edu05, edu09], ignore_index=True)
print("\nDesfechos por ano:", desfechos.groupby("ano").size().to_dict())


Desfechos por ano: {2005: 11499, 2009: 7935}


In [17]:
# ======================================================================
# 3. Juntar desfechos + grupo T/C + covariáveis — SEM duplicar colunas
# ======================================================================
link_sem_chave = link.drop(columns=["w2005_chave"])
painel = desfechos.merge(link_sem_chave, on="cod_dtm", how="inner")

painel["pos"] = (painel["ano"] == 2009).astype(int)
painel["tratado"] = (painel["grupo"] == "T1").astype(int)

# renomear covariáveis para nomes mais limpos
painel = painel.rename(columns={
    "w2005_treat_control": "grupo_ifpri",
    "w2005_a3_reg": "regiao",
    "w2005_a1_uf": "uf",
    "w2005_num_age0_age15": "n_criancas_05",
    "w2005_inc_pc_hh_wunemp": "renda_pc_05",
    "w2005_educ_m": "educ_chefe_m_05",
    "w2005_educ_f": "educ_chefe_f_05",
    "w2005_age_m": "idade_chefe_m_05",
    "w2005_age_f": "idade_chefe_f_05",
    "w2005_sitdom": "sitdom_05",
})

# manter só o essencial — drop dos flags T1/C1/C2 redundantes c/ 'grupo'
painel = painel.drop(columns=["w2005_T1", "w2005_C1", "w2005_C2"])

In [18]:
painel.to_csv("clean/day 04/aula04_painel.csv", index=False, encoding="utf-8")

In [19]:
print("\nExtrato:", painel.shape, "| domicílios únicos:", painel["cod_dtm"].nunique())


Extrato: (19425, 19) | domicílios únicos: 12594


In [20]:
print(painel.groupby(["ano", "grupo"]).size().unstack())
print("\ncolunas:", list(painel.columns))

grupo    C1    C2    T1
ano                    
2005   2382  5731  3386
2009   1772  3661  2493

colunas: ['n_criancas', 'attend_med', 'dropout_med', 'grade_now_med', 'ano', 'cod_dtm', 'grupo_ifpri', 'regiao', 'uf', 'n_criancas_05', 'renda_pc_05', 'educ_chefe_m_05', 'educ_chefe_f_05', 'idade_chefe_m_05', 'idade_chefe_f_05', 'sitdom_05', 'grupo', 'pos', 'tratado']
